# 第9回：前処理をPipelineにまとめる

**今日の問い：数値列とカテゴリ列を、安全に同じモデルへ入れるにはどうするか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 列型ごとの前処理をColumnTransformerで分け、Pipelineへ一体化する
- BaseEstimatorとTransformerMixinで、意味のある自作変換器を書く
- 前処理の選択肢をGridSearchCVの探索対象に含める

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- ColumnTransformer：列ごとに別の前処理を割り当てる仕組み
- 自作変換器：fit/transformを実装した独自の前処理
- get_feature_names_out：変換後の列名を取得するAPI
- パラメータ探索：前処理やモデルの設定を系統的に比較すること
- メモリキャッシュ：共通の前処理計算を使い回す仕組み

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

numeric = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
categorical = ["solvent", "catalyst", "scaffold_group"]
X = df[numeric + categorical]
y = df["active"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


## TRY：列ごとの前処理を組み立てる


In [ ]:
numeric_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="median")),
    ("標準化", StandardScaler()),
])
categorical_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("数値列", numeric_process, numeric),
    ("カテゴリ列", categorical_process, categorical),
])
model = Pipeline([("前処理", preprocess), ("予測", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
print(classification_report(y_valid, model.predict(X_valid), target_names=["非活性", "活性"]))


## 未知カテゴリでも予測できるか


In [ ]:
unknown = X_valid.iloc[[0]].copy()
unknown["solvent"] = "New-Solvent"
print("未知カテゴリを含む予測:", model.predict(unknown)[0])


## CORE深掘り：変換後の列名と列数を確認する

One-Hotで列が増えます。`get_feature_names_out`で変換後の姿を見ます。


In [ ]:
names = model.named_steps["前処理"].get_feature_names_out()
transformed = model.named_steps["前処理"].transform(X_train.head(3))
if hasattr(transformed, "toarray"):
    transformed = transformed.toarray()
print("元の列数:", X_train.shape[1], "→ 変換後:", transformed.shape[1])
pd.DataFrame(transformed, columns=names, index=X_train.head(3).index).iloc[:, :10].round(2)


## CHANGE

数値の欠損補完を`median`から`mean`へ変え、同じ検証データで比べます。変更はPipelineの1か所だけにします。


## DEEP DIVE：自作変換器と前処理の探索

意味のある特徴量を作る自作変換器を書き、前処理そのものをハイパーパラメータとして探索します。


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class ChemRatioFeatures(BaseEstimator, TransformerMixin):
    "分子量あたりのTPSAと、最適温度78℃からの距離を足す自作変換器。"
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X["tpsa_per_mw"] = X["tpsa"] / X["molecular_weight"].replace(0, np.nan)
        X["temp_distance"] = (X["temperature_c"] - 78).abs()
        return X

ChemRatioFeatures().fit_transform(df[["tpsa", "molecular_weight", "temperature_c"]].head()).round(3)


### 前処理の設定をGridSearchで選ぶ

補完戦略のような前処理の選択も、交差検証で選べます。


In [ ]:
from sklearn.model_selection import GridSearchCV

grid_pipe = Pipeline([("前処理", preprocess), ("予測", LogisticRegression(max_iter=1000))])
param_grid = {"前処理__数値列__欠損補完__strategy": ["median", "mean"]}
search = GridSearchCV(grid_pipe, param_grid, cv=5, scoring="f1")
search.fit(X_train, y_train)
print("最良設定:", search.best_params_)
print("最良CV F1:", round(search.best_score_, 3))


## よくある誤り

- 全データ平均で欠損補完する
- カテゴリを意味のない大小関係へ変換する
- 本番の未知カテゴリでエラーになる

## SELF-STUDY（任意・30〜60分）

- 分子量あたりのTPSAを作る自作変換器を書き、Pipelineへ組み込む
- 数値標準化の有無と補完戦略をGridSearchCVで比較する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 自作変換器に最低限必要なメソッドは何か
2. 前処理をPipelineへ入れるとリークがなぜ防げるか
3. get_feature_names_outは何に使うか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
